## Logistic Regression Experiments



In [1]:
from __future__ import annotations

import os
import warnings
from typing import List, Tuple

import numpy as np
os.environ["PANDAS_NO_NUMEXPR"] = "1"
os.environ["PANDAS_NO_BOTTLENECK"] = "1"
from sklearn.datasets import load_digits, load_iris, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

try:
    import pandas as pd

    HAS_PANDAS = True
except Exception:
    pd = None
    HAS_PANDAS = False

warnings.filterwarnings("ignore", category=FutureWarning)


class LogisticRegressionScratch:
    def __init__(self, lr: float = 0.1, n_iter: int = 1000) -> None:
        self.lr = lr
        self.n_iter = n_iter
        self.weights: np.ndarray | None = None
        self.bias: float = 0.0

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionScratch":
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0

        for _ in range(self.n_iter):
            linear_model = X @ self.weights + self.bias
            y_prob = self._sigmoid(linear_model)

            error = y_prob - y
            dw = (X.T @ error) / n_samples
            db = float(error.mean())

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if self.weights is None:
            raise RuntimeError("Model has not been fitted yet.")
        linear_model = X @ self.weights + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        return (probs >= 0.5).astype(int)


def part1_logreg_scratch() -> None:
    print("=== Part 1: Logistic Regression from scratch on synthetic data ===")

    X, y = make_classification(
        n_samples=400,
        n_features=2,
        n_redundant=0,
        n_informative=2,
        n_clusters_per_class=1,
        flip_y=0.03,
        class_sep=1.5,
        random_state=42,
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    model = LogisticRegressionScratch(lr=0.1, n_iter=2000)
    model.fit(X_train, y_train)

    def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        return float((y_true == y_pred).mean())

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    print(f"Train accuracy (scratch): {accuracy(y_train, y_train_pred):.3f}")
    print(f"Test accuracy  (scratch): {accuracy(y_test, y_test_pred):.3f}")
    print()


def load_iris_from_csv_or_sklearn() -> pd.DataFrame:
    possible_paths = ["IRIS.csv", "iris.csv", "data/IRIS.csv", "data/iris.csv"]
    if HAS_PANDAS:
        for path in possible_paths:
            try:
                df = pd.read_csv(path)
                print(f"Loaded Iris data from '{path}'.")
                return df
            except FileNotFoundError:
                continue

    print("IRIS.csv not found in the repo; using sklearn.datasets.load_iris() instead.")
    iris = load_iris(as_frame=True)
    return iris.frame


def prepare_iris_X_y(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    df_copy = df.copy()

    label_candidates = ["species", "Species", "target", "class"]
    label_col = None
    for col in label_candidates:
        if col in df_copy.columns:
            label_col = col
            break

    if label_col is None:
        label_col = df_copy.columns[-1]

    X = df_copy.drop(columns=[label_col]).to_numpy()
    y_raw = df_copy[label_col]

    if not np.issubdtype(y_raw.dtype, np.number):
        y = y_raw.astype("category").cat.codes.to_numpy()
    else:
        y = y_raw.to_numpy()

    return X, y


def part2_multiclass_iris() -> None:
    print("=== Part 2: Multiclass logistic regression on Iris ===")

    df_iris = load_iris_from_csv_or_sklearn()
    X, y = prepare_iris_X_y(df_iris)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    C_values: List[float] = [0.01, 0.1, 1.0, 10.0, 100.0]

    print("C value\tTrain F1 (macro)\tTest F1 (macro)")
    for C in C_values:
        clf = LogisticRegression(
            C=C,
            multi_class="auto",
            solver="lbfgs",
            max_iter=500,
            n_jobs=-1,
        )
        clf.fit(X_train, y_train)

        y_train_pred = clf.predict(X_train)
        y_test_pred = clf.predict(X_test)

        f1_train = f1_score(y_train, y_train_pred, average="macro")
        f1_test = f1_score(y_test, y_test_pred, average="macro")

        print(f"{C:<6.2g}\t{f1_train:>7.3f}\t\t\t{f1_test:>7.3f}")

    print()


def load_digits_from_csv_or_sklearn() -> Tuple[np.ndarray, np.ndarray]:
    possible_paths = ["digits.csv", "Digits.csv", "data/digits.csv", "data/Digits.csv"]
    if HAS_PANDAS:
        for path in possible_paths:
            try:
                df = pd.read_csv(path)
                print(f"Loaded digits data from '{path}'.")
                X = df.iloc[:, :-1].to_numpy()
                y = df.iloc[:, -1].to_numpy()
                return X, y
            except FileNotFoundError:
                continue

    print("digits.csv not found in the repo; using sklearn.datasets.load_digits() instead.")
    digits = load_digits()
    return digits.data, digits.target


def part3_overfitting_digits() -> None:
    print("=== Part 3: Overfitting and regularisation on digits ===")

    X, y = load_digits_from_csv_or_sklearn()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    configs = [
        ("Very large C (almost no regularisation)", "l2", 1e6, "lbfgs"),
        ("Ridge (L2, C=1.0)", "l2", 1.0, "lbfgs"),
        ("Lasso (L1, C=1.0)", "l1", 1.0, "liblinear"),
    ]

    print("Model description")
    print("Train F1 (macro) | Test F1 (macro)")
    print("-----------------------------------")

    for name, penalty, C, solver in configs:
        multi_class = "ovr" if solver == "liblinear" else "auto"

        clf = LogisticRegression(
            penalty=penalty,
            C=C,
            solver=solver,
            multi_class=multi_class,
            max_iter=1000,
        )

        clf.fit(X_train, y_train)

        y_train_pred = clf.predict(X_train)
        y_test_pred = clf.predict(X_test)

        f1_train = f1_score(y_train, y_train_pred, average="macro")
        f1_test = f1_score(y_test, y_test_pred, average="macro")

        print(name)
        print(f"{f1_train:>7.3f} (train) | {f1_test:>7.3f} (test)")
        print()


In [2]:
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

part1_logreg_scratch()
part2_multiclass_iris()
part3_overfitting_digits()


=== Part 1: Logistic Regression from scratch on synthetic data ===
Train accuracy (scratch): 0.943
Test accuracy  (scratch): 0.908

=== Part 2: Multiclass logistic regression on Iris ===
IRIS.csv not found in the repo; using sklearn.datasets.load_iris() instead.
C value	Train F1 (macro)	Test F1 (macro)
0.01  	  0.891			  0.815
0.1   	  0.962			  0.933
1     	  0.971			  0.933
10    	  0.990			  0.955
1e+02 	  1.000			  0.933

=== Part 3: Overfitting and regularisation on digits ===
digits.csv not found in the repo; using sklearn.datasets.load_digits() instead.
Model description
Train F1 (macro) | Test F1 (macro)
-----------------------------------
Very large C (almost no regularisation)
  1.000 (train) |   0.953 (test)

Ridge (L2, C=1.0)
  1.000 (train) |   0.957 (test)

Lasso (L1, C=1.0)
  0.997 (train) |   0.955 (test)



/opt/anaconda3/envs/hetu/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/anaconda3/envs/hetu/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/anaconda3/envs/hetu/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/opt/anaconda3/envs/hetu/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/opt/anaconda3/envs/hetu/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWar